# Qwen System-One v1.1.0 — Fine-tuning (receita Decider 0.8B + multilíngue)

**Base:** `Qwen/Qwen3.5-0.8B-Base` · **Receita:** 1 época, LR 1e-5, warmup 150, readout letter-logits + softmax  
**Diferencial:** prompt no formato do nosso runtime (template primeiro) + multilíngue PT/ES/DE/ZH  
**Licença:** Apache 2.0 (mixture de dados públicos — sem contaminação do Decision Index/JevBench)

Ordem: setup → dados (EN + multilíngue) → treino → calibração T → salva no Drive.

## 0. O que você precisa

| Item | Detalhe |
|---|---|
| **GPU** | A100 40GB (Colab Pro) p/ full-finetune; T4/L4 grátis funciona com QLoRA (célula alternativa) |
| **Tempo** | ~4.5h (GH200 ref) — no Colab A100, uma tarde |
| **Drive** | TUDO cacheado: base Qwen + tasks.pkl + parquets + script — Colab NÃO baixa nada (zero download) |
| **Token HF** | NÃO precisa (tudo já está no Drive) |

In [ ]:
# Célula 1 — setup: Drive + GPU check + instala decider do git
# (idempotente: pode rodar quantas vezes quiser)
# TUDO vem do Drive (base, tasks.pkl, parquets) — ZERO download no Colab.
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive:', e, '— seguindo com disco local (treino será perdido se a sessão cair!)')

import os
GROOT = '/content/drive/MyDrive/qwen-system-one'
USE_DRIVE = os.path.isdir(GROOT)
OUT = (GROOT + '/runs/decider08_full' if USE_DRIVE else '/content/runs/decider08_full')
!mkdir -p $OUT /content/work && cd /content/work

!nvidia-smi --query-gpu=name,memory.total --format=csv | head -n 3
!test -d decider || (tar -xzf /content/drive/MyDrive/qwen-system-one/repo/decider-repo.tar.gz && mv .decider-mirror decider) 2>&1 | tail -n 1
!pip install -q ./decider[train] 2>&1 | tail -n 2
print('OUT =', OUT)
# Verificação offline: base + tasks.pkl + parquets já no Drive?
import os as _os
_chk = {
    'base/model.safetensors': GROOT + '/models/base/model.safetensors',
    'data/tasks.pkl': GROOT + '/data/tasks.pkl',
    'data/train (5 parquets)': GROOT + '/data/train',
}
_allok = True
for _k, _p in _chk.items():
    _ok = _os.path.exists(_p)
    _allok = _allok and _ok
    print(('OK  ' if _ok else 'FALTA') + ' ' + _k)
print('OFFLINE PRONTO' if _allok else 'FALTA ARQUIVO NO DRIVE — veja acima')# VRAM check: full-finetune 0.8B precisa ~25-30GB; QLoRA ~12-15GB
!nvidia-smi --query-gpu=memory.free --format=csv,noheader | head -n 1


## 1. Dados: mixture EN (receita Decider) + multilíngue

A mixture base vem do `decider.data` (~95 datasets públicos, `teacher_data/` já no repo). 
O bloco multilíngue adiciona intent/triagem em **PT/ES/DE/ZH** no formato typed-decision — é o diferencial contra Decider/JPT (English-only) e contra o Laya (colapsa fora do inglês).

⚠️ **Regra de ouro:** NADA do Decision Index frozen suite, NADA do JevBench (público ou sealed), NADA do EnvBench held-out. O index é a prova cega.

In [ ]:
# Célula 2 — monta mixture EN + multilíngue (TUDO do Drive, ZERO download)
# Drive/qwen-system-one contém: models/base/, data/tasks.pkl, data/train/*.parquet
import pickle, json, hashlib, os, shutil, subprocess
from pathlib import Path

GROOT = '/content/drive/MyDrive/qwen-system-one'
DATA = GROOT + '/data/train'
print('DATA =', DATA)
for f in ['minds14-en-US.parquet', 'minds14-de-DE.parquet', 'minds14-es-ES.parquet', 'minds14-pt-PT.parquet', 'minds14-zh-CN.parquet']:
    p = os.path.join(DATA, f)
    print(('OK  ' if os.path.exists(p) else 'FALTA') + ' ' + f)

BASE = 'data/tasks.pkl'
TASKS_DRIVE = GROOT + '/data/tasks.pkl'
!test -d /content/work/decider || (cd /content/work && tar -xzf /content/drive/MyDrive/qwen-system-one/repo/decider-repo.tar.gz && mv .decider-mirror decider) 2>&1 | tail -n 1
# tasks.pkl vem do Drive (sem download). Só gera se faltar nos DOIS lugares.
!if [ -f "{TASKS_DRIVE}" ]; then echo 'tasks.pkl no Drive — copiando (sem download)'; mkdir -p /content/work/decider/data && cp "{TASKS_DRIVE}" /content/work/decider/data/tasks.pkl && ls -lh /content/work/decider/data/tasks.pkl; else echo 'tasks.pkl NÃO está no Drive — gerando (demora ~40min, 1x só)'; fi
!test -f /content/work/decider/data/tasks.pkl || (cd /content/work/decider && python -u -m decider.data.core --out {BASE} 2>&1 | tee /tmp/core.log | grep -E "OK |SKIP |tasks:|ERROR |Traceback" | tail -n 30; echo "--- fim (log completo em /tmp/core.log) ---"; tail -n 3 /tmp/core.log)
# Salva o tasks.pkl no Drive p/ reaproveitar nas próximas sessões
!if [ -f /content/work/decider/data/tasks.pkl ] && [ ! -f "{TASKS_DRIVE}" ]; then cp /content/work/decider/data/tasks.pkl "{TASKS_DRIVE}" && echo 'tasks.pkl salvo no Drive'; fi; ls -lh /content/work/decider/data/tasks.pkl "{TASKS_DRIVE}" 2>&1

# Bloco multilíngue: gera do parquet local (SEM download, SEM datasets) via script do repo
# Script TAMBÉM no Drive (100% offline): sempre sobrescreve (garante versão nova)
!cp /content/drive/MyDrive/qwen-system-one/repo/tools/make_multi_typed.py /content/work/ 2>/dev/null || curl -sL -o /content/work/make_multi_typed.py https://raw.githubusercontent.com/italoalmeida0/qwen-system-one/main/tools/make_multi_typed.py; grep -c by_idx /content/work/make_multi_typed.py
!python /content/work/make_multi_typed.py --src $DATA --out /content/work/multi_typed.jsonl

# Bloco multilíngue gerado acima via make_multi_typed.py (MINDS-14, 5 idiomas).
# Verificação rápida do JSONL gerado:
!wc -l /content/work/multi_typed.jsonl && head -c 400 /content/work/multi_typed.jsonl


## 2. Overlap check (8-gram) — anti-contaminação

Antes de treinar: garante que NENHUM exemplo de treino aparece nos benchmarks (Decision Index frozen, JevBench, EnvBench held-out). O próprio index invalida quem treina na prova.

In [ ]:
# Célula 3 — overlap check (duplicatas internas + tasks.pkl vs multi)
# NOTA: o Decision Index é um Space (scores), não um dataset baixável —
# não há texto de benchmark p/ checar. O que checamos de verdade:
# 1. duplicatas internas do multi_typed.jsonl | 2. multi vs tasks.pkl (EN)
def grams8(text):
    toks = text.lower().split()
    return {' '.join(toks[i:i+8]) for i in range(max(0, len(toks)-7))}

import json, pickle
# 1. tasks.pkl (EN, 968k) -> set de 8-grams (amostra p/ não estourar RAM)
tasks = pickle.load(open("/content/work/decider/data/tasks.pkl","rb"))
print("tasks.pkl tipo:", type(tasks).__name__)
EN_GRAMS = set()
N = 0
items = tasks.values() if isinstance(tasks, dict) else tasks
for group in items:
    for ex in (group if isinstance(group, list) else [group]):
        ctx = getattr(ex, "context", None) or (ex.get("context") if isinstance(ex, dict) else "") or ""
        EN_GRAMS |= grams8(str(ctx))
        N += 1
        if N >= 200000: break
    if N >= 200000: break
print(f"EN 8-grams (amostra {N}):", len(EN_GRAMS))

# 2. multi vs EN + duplicatas internas
bad_en, seen, dup = 0, set(), 0
for line in open("/content/work/multi_typed.jsonl"):
    ex = json.loads(line)
    g = grams8(ex["prompt"])
    if g & EN_GRAMS: bad_en += 1
    h = hash(ex["prompt"])
    if h in seen: dup += 1
    seen.add(h)
print(f"multi vs tasks.pkl EN: {bad_en} (=0 ideal) | duplicatas internas: {dup} (=0 ideal)")

# 3. Escreve multi_clean.jsonl (remove os que colidem com EN + dups) — é ESSE que o treino usa
kept, dropped = 0, 0
seen2 = set()
fw = open("/content/work/multi_clean.jsonl", "w")
for line in open("/content/work/multi_typed.jsonl"):
    ex = json.loads(line)
    g = grams8(ex["prompt"])
    h = hash(ex["prompt"])
    if (g & EN_GRAMS) or (h in seen2):
        dropped += 1
        continue
    seen2.add(h)
    fw.write(line)
    kept += 1
fw.close()
print(f"multi_clean.jsonl: {kept} mantidos, {dropped} removidos")


## 3. Treino (receita Decider, base 0.8B)

`train.sh full` traduzido: 1 época, LR 1e-5, warmup 150, `max_tokens 16384`, `max_options 255`, `schema_first_prob 0.5` (metade schema-first = nossa ordem de prompt), `none_prob 0.1`. Readout: letter-logits no answer slot + softmax.

In [ ]:
# Célula 4 — treino full (A100 80GB). Para T4/L4 grátis, ver célula 4b (QLoRA).
# Base 100% do Drive (ZERO download): copia p/ disco local 1x
!mkdir -p /content/work/base && test -f /content/work/base/model.safetensors || cp /content/drive/MyDrive/qwen-system-one/models/base/* /content/work/base/ && ls -lh /content/work/base/
!cd /content/work/decider && python -u -m decider.data.mixture --base data/tasks.pkl --mode full --out data/mixture_full.pkl --probes data/probes.pkl 2>&1 | tee /tmp/mixture.log | tail -n 5

# INJETA o bloco multilíngue na mixture (formato Example/Q interno — sem isso o treino IGNORA o JSONL!)
!cp /content/drive/MyDrive/qwen-system-one/repo/tools/inject_multi.py /content/work/decider/ 2>/dev/null || curl -sL -o /content/work/decider/inject_multi.py https://raw.githubusercontent.com/italoalmeida0/qwen-system-one/main/tools/inject_multi.py; ls -la /content/work/decider/inject_multi.py
!cd /content/work/decider && python inject_multi.py --mixture data/mixture_full.pkl --jsonl /content/work/multi_clean.jsonl --out data/mixture_full.pkl --task minds14_multi

!cd /content/work/decider && python -m decider.train --model /content/work/base --data data/mixture_full.pkl --out "$OUT" --epochs 1 --lr 1e-5 --warmup 150 --max_tokens 16384 --accum 2 --max_options 255 --max_ctx 16384 --none_prob 0.1 --schema_first_prob 0.5 --eval_every 100000 --eval_limit 150 2>&1 | tee "$OUT/../train_full.log"

In [ ]:
# Célula 4b — ALTERNATIVA QLoRA (T4/L4 grátis, 15GB VRAM). Mais lenta, mesmo formato.
# !pip install -q bitsandbytes peft 2>&1 | tail -n 1
# Mesma mixture, com quant 4-bit + LoRA r=16 em todas as projeções (receita JPT):
# !cd /content/work/decider && python -m decider.train --model /content/work/base --data data/mixture_full.pkl --out "$OUT" --epochs 1 --lr 5e-5 --warmup 150 --qlora --lora_r 16 --max_options 255 --schema_first_prob 0.5 2>&1 | tee "$OUT/../train_qlora.log"

## 4. Calibração da temperatura + eval + export

Calibra T em split held-out (JPT: 1.140; Decider-0.8B: 1.03 — o valor final sai daqui). Depois: merge → ONNX → quant Q4 nos 2 arquivos do runtime.

In [ ]:
# Célula 5 — calibração T + eval + quick-check (tudo Python puro, sem ! com variáveis)
import os, sys, subprocess, json
OUTM = OUT + "/model"
DEC = "/content/work/decider"
sys.path.insert(0, DEC)
os.chdir(DEC)

# 5a. Quick-check: 10 casos de triagem PT/EN (sanity do produto)
print("=== 5a. quick-check (10 casos) ===")
from decider.infer import Decider
d = Decider(OUTM)
cases = [
  ("We were billed twice on the March invoice and want a refund.", "billing"),
  ("The application crashes with a segfault when I open the settings page.", "tech"),
  ("Fui cobrado em duplicidade na minha fatura e quero reembolso.", "billing"),
  ("Me cobraron dos veces en mi factura y quiero un reembolso.", "billing"),
  ("Your service has been down for six hours and nobody answers.", "tech"),
  ("Quero fazer upgrade do meu plano para o empresarial.", "sales"),
  ("Can you send me a quote for the business tier?", "sales"),
  ("Preciso da segunda via do boleto que venceu ontem.", "billing"),
  ("The dashboard shows a 500 error every time I export a report.", "tech"),
  ("We would like to purchase more seats for our account.", "sales"),
]
Q = {
  "question": "Which department should handle this ticket?",
  "options": ["billing", "tech", "sales"],
}
ok = 0
for state, exp in cases:
    r = d.decide(state, [Q])
    got = r[0]["choice"]
    conf = r[0].get("confidence", 0)
    m = "OK" if got == exp else "XX"
    ok += (got == exp)
    print(f"[{m}] got={got:8s} exp={exp:8s} conf={conf:.2f} | {state[:50]}")
print(f"quick-check: {ok}/10 matches")

# 5b. Eval oficial do Decider (in-task vs 0.776 / held-out vs 0.707)
print()
print("=== 5b. evaluate (in-task + held-out) ===")
r = subprocess.run([sys.executable, "-m", "decider.evaluate", "--model", OUTM], capture_output=True, text=True, cwd=DEC)
print(r.stdout[-3000:] if r.stdout else "(sem stdout)")
if r.returncode != 0:
    print("stderr:", r.stderr[-1500:])

# 5c. Calibração T (se decider.temperature existir)
print()
print("=== 5c. calibração T ===")
try:
    from decider import temperature as T
    print("temperature module:", [x for x in dir(T) if not x.startswith("_")][:20])
except Exception as e:
    print("temperature module:", e)

print()
print("=== modelo pronto p/ export ONNX Q4 (célula 6) ===")
print("Modelo:", OUTM)


## 5. Export ONNX Q4 (Fase C) — gera os 2 arquivos do runtime

Converte o modelo treinado (PyTorch) para o formato que o `qwen-serve` lê:
- `embed_tokens_q4.onnx` + `embed_tokens_q4.onnx_data` (lookup de embeddings)
- `decoder_model_merged_q4.onnx` + `decoder_model_merged_q4.onnx_data` (decoder com KV cache)

**Pipeline:** optimum.onnxruntime (exporta com past_*/present_*) → renomeia pro layout do runtime → quantiza INT4 block-wise (group_size=32).

⚠️ **Se o optimum reclamar de ops** (Qwen3.5 é híbrido DeltaNet+atenção), usar o exportador do `onnx-community/Qwen3.5-0.8B-ONNX-OPT` como referência — mesma arquitetura, mesmo layout de I/O. O script tem fallback comentado.

In [ ]:
# Célula 6 — export ONNX Q4 (gera os arquivos do runtime)
# 6a. Instala optimum + onnxruntime (se não estiver)
!pip install -q optimum[onnxruntime] onnxruntime 2>&1 | tail -n 2

# 6b. Copia o script do Drive (ou baixa do git)
!cp /content/drive/MyDrive/qwen-system-one/repo/tools/export_onnx_q4.py /content/work/ 2>/dev/null || curl -sL -o /content/work/export_onnx_q4.py https://raw.githubusercontent.com/italoalmeida0/qwen-system-one/main/tools/export_onnx_q4.py
!ls -la /content/work/export_onnx_q4.py

# 6c. Exporta: modelo treinado -> ONNX Q4 (via optimum-cli, mesmo caminho do onnx-community)
import subprocess, sys
MODELO = OUT + "/model"
OUT_ONNX = GROOT + "/models_v1_1"
print(f"Exportando {MODELO} -> {OUT_ONNX}")
r = subprocess.run([sys.executable, "/content/work/export_onnx_q4.py", "--model", MODELO, "--out", OUT_ONNX])
print("export exit:", r.returncode)

# 6d. Verifica os arquivos gerados
import os
for f in ["embed_tokens_q4.onnx", "embed_tokens_q4.onnx_data", "decoder_model_merged_q4.onnx", "decoder_model_merged_q4.onnx_data"]:
    p = os.path.join(OUT_ONNX, f)
    print(("OK  " if os.path.exists(p) else "FALTA") + f" {f} ({os.path.getsize(p)/1024/1024:.1f}MB)" if os.path.exists(p) else ("FALTA " + f))

print()
print("=== Export completo! ===")
print("Próximo: copiar os 4 arquivos pro runtime (models/) e rodar quick-check + bench")


## 6. Experimento de quantização (opcional)

Q3 **não existe** como tipo nativo no ONNX Runtime (só 2/4/8-bit via `MatMulNBits`) — o Q3 do llama.cpp é esquema GGUF próprio, sem kernel no ORT.

O experimento mede a curva acc vs tamanho vs latência:

| Variante | Bits | Tamanho | Esperado |
|---|---|---|---|
| `fp16` | 16 | ~1.6GB | baseline máximo |
| `quantized` (uint8) | 8 | ~800MB | quase igual fp16 |
| **`q4`** | 4 | **~533MB** | boa (nosso atual) |
| `q2` | 2 | ~270MB | deve cair (modelo 0.8B não tem redundância) |

O `optimum-cli --optimize O2` gera fp16/uint8/q4/q4f16; o Q2 sai via `MatMulNBitsQuantizer(bits=2, group_size=64)` do ORT.

In [ ]:
# Célula 6b — experimento: compara fp16/uint8/q4/q2 (acc vs tamanho vs latência)
import subprocess, sys, os
MODELO = OUT + "/model"
OUT_ONNX = GROOT + "/models_v1_1"
r = subprocess.run([sys.executable, "/content/work/export_onnx_q4.py", "--model", MODELO, "--out", OUT_ONNX, "--keep-variants"])
print("export exit:", r.returncode)

# Gera a variante Q2 via MatMulNBitsQuantizer (bits=2, group_size=64)
print()
print("Gerando Q2 (2-bit) via MatMulNBitsQuantizer...")
from onnxruntime.quantization import QuantFormat, QuantType
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer
import onnx
for src_name in ["decoder_model_merged_q4.onnx", "embed_tokens_q4.onnx"]:
    src = os.path.join(OUT_ONNX, src_name)
    if not os.path.exists(src):
        print(f"  {src_name}: não encontrado, pula")
        continue
    model = onnx.load(src, load_external_data=True)
    q = MatMulNBitsQuantizer(model, bits=2, group_size=64, quant_format=QuantFormat.QOperator)
    q.quantize_model()
    dst = os.path.join(OUT_ONNX, src_name.replace("_q4", "_q2"))
    q.model.save(dst, save_as_external_data=True, all_tensors_to_one_file=True, location=os.path.basename(dst) + "_data")
    print(f"  {src_name} -> {os.path.basename(dst)}")

print()
print("Variantes geradas:")
for f in sorted(os.listdir(OUT_ONNX)):
    if f.endswith((".onnx", ".onnx_data")):
        sz = os.path.getsize(os.path.join(OUT_ONNX, f)) / 1024 / 1024
        print(f"  {f}: {sz:.1f}MB")
print()
print("Próximo: rodar quick-check em cada variante e comparar acc + latência")
